# Imputing Missing Values

**Author:** Tohidul Islam Tareq

Simple, KNN, and iterative imputation examples using a synthetic Titanic-like dataset.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams['figure.figsize'] = (7, 4)


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.linear_model import LogisticRegression, BayesianRidge
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.metrics import accuracy_score

rng = np.random.default_rng(RANDOM_STATE)
n = 450
data = pd.DataFrame({
    'pclass': rng.choice([1, 2, 3], size=n, p=[0.25, 0.25, 0.50]),
    'sex': rng.choice(['female', 'male'], size=n),
    'age': rng.normal(31, 14, size=n).clip(1, 80),
    'fare': rng.gamma(2.5, 25, size=n),
    'embarked': rng.choice(['S', 'C', 'Q'], size=n, p=[0.7, 0.2, 0.1])
})
logit = 1.3*(data.sex == 'female') - 0.7*(data.pclass == 3) - 0.015*data.age + 0.004*data.fare
prob = 1/(1+np.exp(-logit))
data['survived'] = rng.binomial(1, prob)
for col, frac in [('age', 0.18), ('fare', 0.08), ('embarked', 0.05)]:
    data.loc[rng.choice(data.index, int(frac*n), replace=False), col] = np.nan

data.head()


,pclass,sex,age,fare,embarked,survived
0,3,female,33.301470,98.477326,S,1
1,2,female,25.533505,107.303783,S,1
2,3,female,56.869550,NaN,Q,0
3,3,male,28.561582,22.989125,NaN,1
4,1,female,NaN,18.024896,S,1


In [3]:
X = data.drop(columns='survived')
y = data['survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)

num_features = ['pclass', 'age', 'fare']
cat_features = ['sex', 'embarked']

def build_pipeline(num_imputer):
    preprocessor = ColumnTransformer([
        ('num', Pipeline([('imputer', num_imputer), ('scaler', StandardScaler())]), num_features),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_features)
    ])
    return Pipeline([('preprocess', preprocessor), ('model', LogisticRegression(max_iter=1000))])

pipelines = {
    'mean_imputer': build_pipeline(SimpleImputer(strategy='mean')),
    'knn_imputer': build_pipeline(KNNImputer(n_neighbors=5)),
    'iterative_imputer': build_pipeline(IterativeImputer(estimator=BayesianRidge(), random_state=RANDOM_STATE, max_iter=10))
}

scores = []
for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    scores.append({'method': name, 'accuracy': accuracy_score(y_test, pred)})
pd.DataFrame(scores)


,method,accuracy
0,mean_imputer,0.646018
1,knn_imputer,0.592920
2,iterative_imputer,0.628319
